# Baseline Interpolation

In [1]:
# ============================================
# STEP 5.1 — Import Libraries & Define Paths
# ============================================

import pandas as pd
import numpy as np
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

# Raw data folder
RAW_DATA = PROJECT_ROOT / "Data" / "raw data"

# Processed data folder
PROCESSED_DATA = PROJECT_ROOT / "Data" / "processed"

# Dataset path
AIR_FILE = RAW_DATA / "air_quality_2019_2021_hourly.csv"
MET_FILE = RAW_DATA / "hourly_meteorology_16stations_2019_2021.csv"

# Fixed artificial mask created in STEP 3
MASK_FILE = PROCESSED_DATA / "fixed_artificial_mask_new.csv"

print("Project Root:")
print(PROJECT_ROOT)

print("\nAir Quality File:")
print(AIR_FILE)

print("\nMeteorology File:")
print(MET_FILE)

print("\nFixed Mask File:")
print(MASK_FILE)

Project Root:
e:\Edge_air_pollution imputation

Air Quality File:
e:\Edge_air_pollution imputation\Data\raw data\air_quality_2019_2021_hourly.csv

Meteorology File:
e:\Edge_air_pollution imputation\Data\raw data\hourly_meteorology_16stations_2019_2021.csv

Fixed Mask File:
e:\Edge_air_pollution imputation\Data\processed\fixed_artificial_mask_new.csv


In [ ]:
# ============================================
# STEP 5.2 — Load Dataset & Fixed Mask
# ============================================

# Load raw datasets
air_df = pd.read_csv(AIR_FILE)
met_df = pd.read_csv(MET_FILE)

# Convert timestamp
air_df["timestamp"] = pd.to_datetime(air_df["timestamp"])
met_df["timestamp"] = pd.to_datetime(met_df["timestamp"])

# Merge Air Quality + Meteorology
merged_df = pd.merge(
    air_df,
    met_df,
    on=["station_id", "station_name", "timestamp"],
    how="inner"
)

# Load fixed artificial mask
fixed_mask = pd.read_csv(MASK_FILE)

print("Air Quality shape:", air_df.shape)
print("Meteorology shape:", met_df.shape)
print("Merged dataset shape:", merged_df.shape)
print("Fixed mask shape:", fixed_mask.shape)

Air Quality shape: (420864, 10)
Meteorology shape: (420864, 9)
Merged dataset shape: (420864, 16)
Fixed mask shape: (420864, 13)


In [3]:
# ============================================
# STEP 5.3 — Define & Verify 13 Features
# ============================================

pollution_features = [
    "pm25",
    "pm10",
    "no2",
    "o3",
    "so2",
    "nox",
    "co"
]

weather_features = [
    "temperature",
    "relative_humidity",
    "wind_speed",
    "wind_direction",
    "pressure",
    "rainfall"
]

feature_columns = pollution_features + weather_features

print("Total features:", len(feature_columns))

print("\nFeature columns:")
for i, feature in enumerate(feature_columns, start=1):
    print(f"{i:2}. {feature}")

print("\nMerged dataset feature columns present:",
      all(feature in merged_df.columns for feature in feature_columns))

print("Fixed mask columns match:",
      list(fixed_mask.columns) == feature_columns)

Total features: 13

Feature columns:
 1. pm25
 2. pm10
 3. no2
 4. o3
 5. so2
 6. nox
 7. co
 8. temperature
 9. relative_humidity
10. wind_speed
11. wind_direction
12. pressure
13. rainfall

Merged dataset feature columns present: True
Fixed mask columns match: True


In [4]:
# ============================================
# STEP 5.4 — Create Ground Truth & Masked Data
# ============================================

# Original data as ground truth
ground_truth_df = merged_df.copy()

# Copy for interpolation input
masked_df = merged_df.copy()

# Apply fixed artificial mask
# 1 = available, 0 = masked
for feature in feature_columns:
    masked_df.loc[fixed_mask[feature] == 0, feature] = np.nan

print("Ground truth shape:", ground_truth_df.shape)
print("Masked data shape:", masked_df.shape)

# Count NaN values in model features
print("\nNaN count in masked data:")
print(masked_df[feature_columns].isna().sum().sum())

Ground truth shape: (420864, 16)
Masked data shape: (420864, 16)

NaN count in masked data:
1321566


In [5]:
# ============================================
# STEP 5.5 — Sort Data Station-wise & Time-wise
# ============================================

# Sort both datasets using the same station and time order
ground_truth_df = ground_truth_df.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

masked_df = masked_df.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

print("Ground truth sorted:", ground_truth_df.shape)
print("Masked data sorted:", masked_df.shape)

# Verify first and last timestamps
print("\nFirst row:")
print(masked_df[["station_id", "timestamp"]].head(1))

print("\nLast row:")
print(masked_df[["station_id", "timestamp"]].tail(1))

Ground truth sorted: (420864, 16)
Masked data sorted: (420864, 16)

First row:
   station_id  timestamp
0          66 2019-01-01

Last row:
        station_id           timestamp
420863          83 2021-12-31 23:00:00


In [9]:
# ============================================
# STEP 5.6 — Verify Fixed Mask Alignment
# ============================================

print("Merged data shape:", merged_df.shape)
print("Fixed mask shape:", fixed_mask.shape)

# Check whether the row counts match
if len(merged_df) != len(fixed_mask):
    raise ValueError(
        "Row count mismatch: merged data and fixed mask "
        "do not have the same number of rows."
    )

# Check that the mask contains only 0 and 1
mask_values = fixed_mask[feature_columns].to_numpy()

print("\nUnique mask values:", np.unique(mask_values))
print("Mask contains missing values:", fixed_mask[feature_columns].isna().any().any())

if not np.isin(mask_values, [0, 1]).all():
    raise ValueError("The fixed mask must contain only 0 and 1.")

print("\nBasic mask validation passed.")
print("Note: matching row counts alone do not prove row alignment.")

Merged data shape: (420864, 16)
Fixed mask shape: (420864, 13)

Unique mask values: [0 1]
Mask contains missing values: False

Basic mask validation passed.
Note: matching row counts alone do not prove row alignment.


In [10]:
# ============================================
# STEP 5.7 — Check Mask Row Alignment
# ============================================

# Display the first few rows before sorting
print("First 5 rows of merged data:")
print(
    merged_df[
        ["station_id", "station_name", "timestamp"]
    ].head()
)

print("\nFirst 5 rows of fixed mask:")
print(fixed_mask[feature_columns].head())

# Check whether the merged data is already sorted
merged_is_sorted = (
    merged_df[["station_id", "timestamp"]]
    .equals(
        merged_df[["station_id", "timestamp"]]
        .sort_values(["station_id", "timestamp"])
        .reset_index(drop=True)
    )
)

print("\nMerged data already sorted:", merged_is_sorted)

# Check for duplicate station-timestamp combinations
duplicate_count = merged_df.duplicated(
    subset=["station_id", "timestamp"]
).sum()

print("Duplicate station-timestamp rows:", duplicate_count)

First 5 rows of merged data:
   station_id  station_name           timestamp
0          71  CAUSEWAY BAY 2019-01-01 00:00:00
1          71  CAUSEWAY BAY 2019-01-01 01:00:00
2          71  CAUSEWAY BAY 2019-01-01 02:00:00
3          71  CAUSEWAY BAY 2019-01-01 03:00:00
4          71  CAUSEWAY BAY 2019-01-01 04:00:00

First 5 rows of fixed mask:
   pm25  pm10  no2  o3  so2  nox  co  temperature  relative_humidity  \
0     1     1    1   1    0    0   0            1                  1   
1     1     0    0   1    1    1   1            1                  0   
2     0     1    1   0    1    0   0            1                  1   
3     0     0    1   0    1    1   1            1                  1   
4     0     0    1   1    0    0   0            1                  1   

   wind_speed  wind_direction  pressure  rainfall  
0           1               0         1         1  
1           1               1         1         1  
2           1               1         0         1  
3           1

In [6]:
# ============================================
# STEP 5.6 — Station-wise Linear Interpolation
# ============================================

interpolated_df = masked_df.copy()

# Interpolate each feature separately for each station
for feature in feature_columns:
    interpolated_df[feature] = (
        interpolated_df
        .groupby("station_id")[feature]
        .transform(
            lambda x: x.interpolate(
                method="linear",
                limit_direction="both"
            )
        )
    )

print("Interpolation completed.")

print("\nRemaining NaN values:")
print(interpolated_df[feature_columns].isna().sum().sum())

Interpolation completed.

Remaining NaN values:
210432


In [7]:
# ============================================
# STEP 5.7 — Verify Interpolated Values
# ============================================

# Artificially masked positions:
# Original mask = 1 and fixed mask = 0
artificial_mask_positions = (
    (merged_df[feature_columns].notna()) &
    (fixed_mask[feature_columns] == 0)
)

# Count how many artificial positions were created
total_artificial_positions = artificial_mask_positions.sum().sum()

# Count how many of those positions now have interpolated values
filled_artificial_positions = (
    interpolated_df[feature_columns]
    .notna() &
    artificial_mask_positions
).sum().sum()

# Count artificial positions still remaining NaN
remaining_artificial_nan = (
    interpolated_df[feature_columns].isna() &
    artificial_mask_positions
).sum().sum()

print("Total artificial masked positions:", total_artificial_positions)
print("Successfully interpolated:", filled_artificial_positions)
print("Still NaN:", remaining_artificial_nan)

Total artificial masked positions: 1037455
Successfully interpolated: 1011602
Still NaN: 25853


In [6]:

# ============================================
# STEP 5.2 — Load Dataset & Fixed Mask
# ============================================

import pandas as pd
import numpy as np
from pathlib import Path

# Get the project root directory
PROJECT_ROOT = Path.cwd().parent

# Define correct file paths
AIR_FILE = PROJECT_ROOT / "Data" / "raw data" / "air_quality_2019_2021_hourly.csv"
MET_FILE = PROJECT_ROOT / "Data" / "raw data" / "hourly_meteorology_16stations_2019_2021.csv"
MASK_FILE = PROJECT_ROOT / "Data" / "processed" / "fixed_artificial_mask_new.csv"

# Verify that all files exist
for file_path in [AIR_FILE, MET_FILE, MASK_FILE]:
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

# Load raw datasets
air_df = pd.read_csv(AIR_FILE)
met_df = pd.read_csv(MET_FILE)

# Convert timestamps
air_df["timestamp"] = pd.to_datetime(air_df["timestamp"])
met_df["timestamp"] = pd.to_datetime(met_df["timestamp"])

# Merge air quality and meteorology
merged_df = pd.merge(
    air_df,
    met_df,
    on=["station_id", "station_name", "timestamp"],
    how="inner",
    validate="one_to_one"
)

# Load the existing fixed mask without modifying it
fixed_mask = pd.read_csv(MASK_FILE)

# Display results
print("Project root:", PROJECT_ROOT)
print("Air Quality shape:", air_df.shape)
print("Meteorology shape:", met_df.shape)
print("Merged dataset shape:", merged_df.shape)
print("Fixed mask shape:", fixed_mask.shape)

print("\nAll required files loaded successfully.")


Project root: e:\Edge_air_pollution imputation
Air Quality shape: (420864, 10)
Meteorology shape: (420864, 9)
Merged dataset shape: (420864, 16)
Fixed mask shape: (420864, 13)

All required files loaded successfully.


In [8]:

# ============================================
# STEP 5.5 — Create Ground Truth and Masked Data
# ============================================

import pandas as pd
import numpy as np

# Define the 13 feature columns
feature_columns = [
    "pm25", "pm10", "no2", "o3", "so2", "nox", "co",
    "temperature", "relative_humidity", "wind_speed",
    "wind_direction", "pressure", "rainfall"
]

# Create ground-truth data from the merged dataset
ground_truth_df = merged_df[
    ["station_id", "station_name", "timestamp"] + feature_columns
].copy()

# Sort ground-truth data by station and timestamp
ground_truth_df = ground_truth_df.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

print("Ground truth created:", ground_truth_df.shape)
print("Ground truth first station:",
      ground_truth_df["station_id"].iloc[0])

# Check whether the fixed mask has the expected dimensions
assert fixed_mask.shape == (
    len(merged_df), len(feature_columns)
), "Fixed mask dimensions do not match the expected dimensions."

print("Fixed mask dimensions verified:", fixed_mask.shape)

Ground truth created: (420864, 16)
Ground truth first station: 66
Fixed mask dimensions verified: (420864, 13)


In [11]:
# ============================================
# STEP 5.8 — Check Data and Mask Row Ordering
# ============================================

print("Merged dataset — first and last rows:")
print(
    merged_df[["station_id", "timestamp"]]
    .iloc[[0, -1]]
)

print("\nFixed mask — first and last rows:")
print(fixed_mask[feature_columns].iloc[[0, -1]])

print("\nMerged data station order:")
print(merged_df["station_id"].drop_duplicates().tolist())

print("\nMerged data sorted by station and timestamp:",
      merged_df[["station_id", "timestamp"]].reset_index(drop=True).equals(
          merged_df.sort_values(
              ["station_id", "timestamp"]
          )[["station_id", "timestamp"]].reset_index(drop=True)
      ))

Merged dataset — first and last rows:
        station_id           timestamp
0               71 2019-01-01 00:00:00
420863          70 2021-12-31 23:00:00

Fixed mask — first and last rows:
        pm25  pm10  no2  o3  so2  nox  co  temperature  relative_humidity  \
0          1     1    1   1    0    0   0            1                  1   
420863     1     1    1   1    1    1   1            1                  0   

        wind_speed  wind_direction  pressure  rainfall  
0                1               0         1         1  
420863           1               1         0         1  

Merged data station order:
[71, 79, 80, 73, 72, 74, 81, 66, 75, 69, 76, 83, 77, 82, 78, 70]

Merged data sorted by station and timestamp: False


In [12]:

# ============================================
# STEP 5.8A — Align and Validate Fixed Mask
# ============================================

import numpy as np
import pandas as pd

# Define the expected feature order
feature_columns = [
    "pm25", "pm10", "no2", "o3", "so2", "nox", "co",
    "temperature", "relative_humidity", "wind_speed",
    "wind_direction", "pressure", "rainfall"
]

# STEP 1: Check required variables
required_variables = ["merged_df", "fixed_mask"]

for variable in required_variables:
    if variable not in globals():
        raise NameError(
            f"{variable} is not defined. Run STEP 5.2 first."
        )

# STEP 2: Validate data dimensions
print("Merged dataset shape:", merged_df.shape)
print("Fixed mask shape:", fixed_mask.shape)

assert len(fixed_mask) == len(merged_df), (
    "ERROR: Fixed mask and merged dataset have different row counts"
)

Merged dataset shape: (420864, 16)
Fixed mask shape: (420864, 13)


In [ ]:
# ============================================
# STEP 5.8B — Calculate MAE & RMSE
# ============================================

from sklearn.metrics import mean_absolute_error, mean_squared_error

results = []

for feature in feature_columns:

    # Artificially masked positions
    artificial_positions = (
        ground_truth_df[feature].notna() &
        (fixed_mask_sorted[feature] == 0)
    )

    # Interpolation must have produced a value
    interpolation_available = (
        interpolated_df[feature].notna()
    )

    # Final valid evaluation positions
    valid_positions = (
        artificial_positions &
        interpolation_available
    )

    # Ground truth
    y_true = ground_truth_df.loc[
        valid_positions, feature
    ]

    # Interpolated prediction
    y_pred = interpolated_df.loc[
        valid_positions, feature
    ]

    # Extra safety: remove any remaining NaN
    valid_numeric = (
        y_true.notna() &
        y_pred.notna()
    )

    y_true = y_true[valid_numeric]
    y_pred = y_pred[valid_numeric]

    # Metrics
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    results.append({
        "Feature": feature,
        "Evaluation_Points": len(y_true),
        "MAE": mae,
        "RMSE": rmse
    })

# Create final results table
interpolation_results = pd.DataFrame(results)

print(interpolation_results.to_string(index=False))

          Feature  Evaluation_Points       MAE      RMSE
             pm25              81706  1.928010  2.811731
             pm10              82146  2.660454  3.965873
              no2              81488  5.481378  8.302123
               o3              82138  5.498488  8.463845
              so2              82128  0.524086  0.950927
              nox              76520 13.893732 26.335951
               co              46323  3.506766  5.954657
      temperature              84162  0.250113  0.373373
relative_humidity              84296  1.491802  2.309094
       wind_speed              84069  0.298734  0.435866
   wind_direction              84344 12.304220 37.624868
         pressure              83978  0.206277  0.274462
         rainfall              84157  0.167156  0.710222


In [12]:
# ============================================
# STEP 5.9 — Save Interpolation Baseline Results
# ============================================

# Output file path
INTERPOLATION_RESULTS_FILE = (
    PROCESSED_DATA / "interpolation_baseline_results_new.csv"
)

# Save results
interpolation_results.to_csv(
    INTERPOLATION_RESULTS_FILE,
    index=False
)

print("Interpolation baseline results saved successfully!")
print("\nFile:")
print(INTERPOLATION_RESULTS_FILE)

print("\nSaved results:")
print(interpolation_results)

Interpolation baseline results saved successfully!

File:
e:\Edge_air_pollution imputation\Data\processed\interpolation_baseline_results_new.csv

Saved results:
              Feature  Evaluation_Points        MAE       RMSE
0                pm25              81706   1.928010   2.811731
1                pm10              82146   2.660454   3.965873
2                 no2              81488   5.481378   8.302123
3                  o3              82138   5.498488   8.463845
4                 so2              82128   0.524086   0.950927
5                 nox              76520  13.893732  26.335951
6                  co              46323   3.506766   5.954657
7         temperature              84162   0.250113   0.373373
8   relative_humidity              84296   1.491802   2.309094
9          wind_speed              84069   0.298734   0.435866
10     wind_direction              84344  12.304220  37.624868
11           pressure              83978   0.206277   0.274462
12           rainfal

In [5]:
from pathlib import Path

# Show the notebook's current working directory
print("Current working directory:")
print(Path.cwd())

# Search for the required dataset files inside the project
project_root = Path(r"E:\Edge_air_pollution imputation")

print("\nSearching for air-quality dataset...")
for file in project_root.rglob("air_quality_2019_2021_hourly.csv"):
    print(file)

print("\nSearching for meteorology dataset...")
for file in project_root.rglob("hourly_meteorology_16stations_2019_2021.csv"):
    print(file)

print("\nSearching for fixed mask...")
for file in project_root.rglob("fixed_artificial_mask_new.csv"):
    print(file)

Current working directory:
e:\Edge_air_pollution imputation\Notebook

Searching for air-quality dataset...
E:\Edge_air_pollution imputation\Data\raw data\air_quality_2019_2021_hourly.csv

Searching for meteorology dataset...
E:\Edge_air_pollution imputation\Data\raw data\hourly_meteorology_16stations_2019_2021.csv

Searching for fixed mask...
E:\Edge_air_pollution imputation\Data\processed\fixed_artificial_mask_new.csv


In [13]:
# ============================================
# STEP 5.9 — Inspect Station-wise Row Ordering
# ============================================

# Identify the row positions needed to sort merged_df.
sort_order = merged_df.sort_values(
    ["station_id", "timestamp"]
).index.to_numpy()

print("Total rows:", len(merged_df))
print("Rows whose positions change during sorting:",
      int((sort_order != np.arange(len(sort_order))).sum()))

print("\nOriginal station order:")
print(merged_df["station_id"].drop_duplicates().tolist())

sorted_merged_df = merged_df.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

print("\nSorted station order:")
print(sorted_merged_df["station_id"].drop_duplicates().tolist())

# Confirm that sorting does not change the number of rows.
assert len(sorted_merged_df) == len(fixed_mask)

print("\nSorting inspection completed.")
print("Original mask remains unchanged.")

Total rows: 420864
Rows whose positions change during sorting: 394560

Original station order:
[71, 79, 80, 73, 72, 74, 81, 66, 75, 69, 76, 83, 77, 82, 78, 70]

Sorted station order:
[66, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83]

Sorting inspection completed.
Original mask remains unchanged.


In [14]:
# ============================================
# STEP 5.10 — Verify Mask Before Application
# ============================================

# Check that the original merged dataset and mask have equal row counts
assert len(merged_df) == len(fixed_mask)

# Calculate how many entries are masked (0) and available (1)
mask_values = fixed_mask[feature_columns].to_numpy()

unique_values, counts = np.unique(
    mask_values,
    return_counts=True
)

print("Original mask statistics:")
for value, count in zip(unique_values, counts):
    label = "Available" if value == 1 else "Masked"
    print(f"{label} ({value}): {count:,}")

# Inspect the first five rows without modifying the mask
print("\nFirst five rows of the original mask:")
print(fixed_mask[feature_columns].head())

# Verify the dimensions
print("\nMerged data shape:", merged_df.shape)
print("Original mask shape:", fixed_mask.shape)

print("\nOriginal mask remains unchanged.")

Original mask statistics:
Masked (0): 1,321,566
Available (1): 4,149,666

First five rows of the original mask:
   pm25  pm10  no2  o3  so2  nox  co  temperature  relative_humidity  \
0     1     1    1   1    0    0   0            1                  1   
1     1     0    0   1    1    1   1            1                  0   
2     0     1    1   0    1    0   0            1                  1   
3     0     0    1   0    1    1   1            1                  1   
4     0     0    1   1    0    0   0            1                  1   

   wind_speed  wind_direction  pressure  rainfall  
0           1               0         1         1  
1           1               1         1         1  
2           1               1         0         1  
3           1               0         1         1  
4           1               1         1         1  

Merged data shape: (420864, 16)
Original mask shape: (420864, 13)

Original mask remains unchanged.
